# Hospital Dataset Cleaning & Preprocessing

## Objective

Prepare a raw hospital dataset for Machine Learning.

## Preprocessing Tasks

1. Load dataset
2. Understand dataset
3. Detect missing values
4. Handle missing values
5. Encode categorical features
6. Split features and target
7. Train-test split
8. Scale numerical features
9. Feature selection
10. Build preprocessing pipeline
11. Save processed data

## 1. Importing Libraries

In [31]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression

print("Libraries are successfully imported.")

Libraries are successfully imported.


## 2. Loading Dataset

In [32]:
df= pd.read_csv("../data/raw_dataset/hypertension_dataset.csv")
df.head(5)
df.shape
df.columns
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 1985 entries, 0 to 1984
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Age               1985 non-null   int64  
 1   Salt_Intake       1985 non-null   float64
 2   Stress_Score      1985 non-null   int64  
 3   BP_History        1985 non-null   str    
 4   Sleep_Duration    1985 non-null   float64
 5   BMI               1985 non-null   float64
 6   Medication        1186 non-null   str    
 7   Family_History    1985 non-null   str    
 8   Exercise_Level    1985 non-null   str    
 9   Smoking_Status    1985 non-null   str    
 10  Has_Hypertension  1985 non-null   str    
dtypes: float64(3), int64(2), str(6)
memory usage: 239.5 KB


,Age,Salt_Intake,Stress_Score,Sleep_Duration,BMI
count,1985.000000,1985.000000,1985.000000,1985.000000,1985.000000
mean,50.341058,8.531688,4.979345,6.452242,26.015315
std,19.442042,1.994907,3.142303,1.542207,4.512857
min,18.000000,2.500000,0.000000,1.500000,11.900000
25%,34.000000,7.200000,2.000000,5.400000,23.000000
50%,50.000000,8.500000,5.000000,6.500000,25.900000
75%,67.000000,9.900000,8.000000,7.500000,29.100000
max,84.000000,16.400000,10.000000,11.400000,41.900000


## 3. Detecting Missing Values

In [33]:
df.isnull().sum()
# missing_percentage=(
#     df.isnull().sum() / len(df)
# ) * 100
df["Medication"].value_counts(dropna=False)
df.duplicated().sum()


np.int64(0)

## 4. Seperate Features and Target

In [34]:
x= df.drop("Has_Hypertension", axis=1)
y= df["Has_Hypertension"]

## 5. Encode Target Variable

In [35]:
y=y.map({
    "No": 0,
    "Yes": 1
})
y.head()

0    1
1    0
2    0
3    1
4    0
Name: Has_Hypertension, dtype: int64

## 6. Split dataset into Training and Testing sets

In [36]:
x_train, x_test, y_train, y_test= train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Training features:", x_train.shape)
print("Testing features:", x_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (1588, 10)
Testing features: (397, 10)
Training target: (1588,)
Testing target: (397,)


## 7. Numerical and Categorical Features

In [37]:
numerical_features=[
    "Age","Salt_Intake", "Stress_Score", "Sleep_Duration", "BMI"
]
categorical_features=[
    "BP_History", "Medication", "Family_History", "Exercise_Level", "Smoking_Status"
]

## 8. Numerical Preprocessing Pipeline

In [38]:
numerical_pipeline=Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

## 9. Categorical Preprocessing Pipeline

In [39]:
categorical_pipeline= Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

## 10. Combine Preprocessing Pipelines

In [40]:
preprocessor= ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

## 11. Fit and transform Data and also Transform testing data

In [41]:
x_train_processed= preprocessor.fit_transform(x_train)
x_train_processed.shape
x_test_processed= preprocessor.transform(x_test)
x_test_processed.shape

(397, 19)

## 12. Get processd Feature Names

In [42]:
feature_names= preprocessor.get_feature_names_out()
feature_names

array(['num__Age', 'num__Salt_Intake', 'num__Stress_Score',
       'num__Sleep_Duration', 'num__BMI', 'cat__BP_History_Hypertension',
       'cat__BP_History_Normal', 'cat__BP_History_Prehypertension',
       'cat__Medication_ACE Inhibitor', 'cat__Medication_Beta Blocker',
       'cat__Medication_Diuretic', 'cat__Medication_Other',
       'cat__Family_History_No', 'cat__Family_History_Yes',
       'cat__Exercise_Level_High', 'cat__Exercise_Level_Low',
       'cat__Exercise_Level_Moderate', 'cat__Smoking_Status_Non-Smoker',
       'cat__Smoking_Status_Smoker'], dtype=object)

## 13. Convert processed training data into a dataframe

In [43]:
x_train_processed_df= pd.DataFrame(
    x_train_processed,
    columns=feature_names,
    index= x_train.index
)
x_train_processed_df.head()

,num__Age,num__Salt_Intake,num__Stress_Score,num__Sleep_Duration,num__BMI,cat__BP_History_Hypertension,cat__BP_History_Normal,cat__BP_History_Prehypertension,cat__Medication_ACE Inhibitor,cat__Medication_Beta Blocker,cat__Medication_Diuretic,cat__Medication_Other,cat__Family_History_No,cat__Family_History_Yes,cat__Exercise_Level_High,cat__Exercise_Level_Low,cat__Exercise_Level_Moderate,cat__Smoking_Status_Non-Smoker,cat__Smoking_Status_Smoker
1168,1.057045,-0.613890,1.291772,0.711610,0.168521,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
1680,-0.944756,-0.218611,-0.299396,0.711610,0.765425,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
115,-1.047413,-0.218611,-1.254097,0.843747,0.898070,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1602,0.184465,-0.959759,1.291772,-0.940104,-1.135824,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
1159,0.646419,-2.244415,0.655305,0.645541,-0.981071,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0


## 14. Check Missing Values After Preprocessing

In [44]:
x_train_processed_df.isnull().sum().sum()

np.int64(0)

## 15. Inspect Scaled Numerical Features and One-Hot Encoded Features

In [45]:
x_train_processed_df.filter(
    like="num__"
).describe()
x_train_processed_df.filter(
    like="cat__"
).head()

,cat__BP_History_Hypertension,cat__BP_History_Normal,cat__BP_History_Prehypertension,cat__Medication_ACE Inhibitor,cat__Medication_Beta Blocker,cat__Medication_Diuretic,cat__Medication_Other,cat__Family_History_No,cat__Family_History_Yes,cat__Exercise_Level_High,cat__Exercise_Level_Low,cat__Exercise_Level_Moderate,cat__Smoking_Status_Non-Smoker,cat__Smoking_Status_Smoker
1168,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
1680,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
115,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1602,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
1159,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0


## 16. Normalization

In [46]:
minmax_scaler= MinMaxScaler()
age_normalized= minmax_scaler.fit_transform(
    x_train[["Age"]]
)
age_normalized[:5]

array([[0.8030303 ],
       [0.21212121],
       [0.18181818],
       [0.54545455],
       [0.68181818]])

## 17. Feature Selection

In [47]:
feature_selection_preprocessor= ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

x_train_selection=(
    feature_selection_preprocessor.fit_transform(x_train)
)
feature_names_for_selection=(
    feature_selection_preprocessor.get_feature_names_out()
)
selector= SelectKBest(
    score_func= f_regression,
    k= 5
)
x_train_selected= selector.fit_transform(x_train_selection, y_train)
x_train_selected.shape

selected_features= feature_names_for_selection[
    selector.get_support()
]
selected_features

array(['cat__BP_History_Hypertension', 'cat__BP_History_Normal',
       'cat__Family_History_No', 'cat__Family_History_Yes',
       'cat__Smoking_Status_Smoker'], dtype=object)

## 18. Create the complete ML pipeline

In [48]:
from sklearn.linear_model import LogisticRegression
model=Pipeline([
    ("processing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

## 19. Train the model

In [49]:
model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('processing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers

## 20. Make Predictions

In [50]:
y_pred= model.predict(x_test)
y_pred[:10]

array([1, 1, 0, 1, 0, 1, 1, 1, 0, 1])

## 21. Compare Actual and Predicated values

In [51]:
comparison= pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})
comparison.head(10)

,Actual,Predicted
0,1,1
1,1,1
2,0,0
3,1,1
4,0,0
5,0,1
6,0,1
7,0,1
8,0,0
9,1,1


## 22. Evaluate Model Accuracy

In [52]:
accuracy= model.score(x_test, y_test)
print("Accuracy: ", accuracy)

Accuracy:  0.8740554156171285


## 23. Classification Report and confusion matrix

In [53]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred))
cm= confusion_matrix(y_test, y_pred)
cm

              precision    recall  f1-score   support

           0       0.86      0.88      0.87       191
           1       0.89      0.86      0.88       206

    accuracy                           0.87       397
   macro avg       0.87      0.87      0.87       397
weighted avg       0.87      0.87      0.87       397



array([[169,  22],
       [ 28, 178]])

## 24. Confusion Metrix Visualization

In [54]:
fig= px.imshow(
    cm, text_auto=True,
    x=["No", "Yes"],
    y=["No", "Yes"],
    labels={
        "x": "Predicated",
        "y": "Actual",
        "color": "Count"
    }, title="Confusion Matrix"
)
fig.show()
fig.write_image("../outputs/confusion_matrix.png")

## 25. Precision, Recall, and F1-score

In [55]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

Accuracy : 0.8740554156171285
Precision: 0.89
Recall   : 0.8640776699029126
F1 Score : 0.8768472906403941


## 26. Save the Trained Pipeline

In [58]:
import joblib
import os
os.makedirs("../models", exist_ok=True)

joblib.dump(model, "../models/hypertension_model.pkl")

['../models/hypertension_model.pkl']

## 27. Load the saved pipeline and test

In [60]:
loaded_model = joblib.load(
    "../models/hypertension_model.pkl"
)
loaded_predictions = loaded_model.predict(x_test)

loaded_predictions[:10]
loaded_accuracy = loaded_model.score(
    x_test,
    y_test
)

print("Loaded Model Accuracy:", loaded_accuracy)

Loaded Model Accuracy: 0.8740554156171285
